In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# מעברי AI Transpiler
מעברי ה-Transpiler המבוססים על בינה מלאכותית הם מעברים שמשמשים כתחליף ישיר למעברי Qiskit "המסורתיים" עבור משימות transpiling מסוימות. הם לרוב מניבים תוצאות טובות יותר מאלגוריתמים היוריסטיים קיימים (כמו עומק נמוך יותר ומספר שערי CNOT קטן יותר), אך גם מהירים בהרבה מאלגוריתמי אופטימיזציה כמו פותרי Boolean satisfiability. מעברי ה-AI Transpiler יכולים לרוץ בסביבה המקומית שלך או בענן באמצעות שירות Qiskit Transpiler Service אם אתה חלק מ-IBM Quantum&reg; Premium Plan, Flex Plan, או On-Prem (דרך IBM Quantum Platform API) Plan.

> **Note:** מעברי ה-Transpiler המבוססים על בינה מלאכותית נמצאים בסטטוס שחרור בטא, וכפופים לשינויים.
>     אם יש לך משוב או שאתה רוצה ליצור קשר עם צוות המפתחים, אנא השתמש ב[ערוץ Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) הזה.

המעברים הבאים זמינים כרגע:

**מעברי Routing**
 - `AIRouting`: בחירת פריסה (Layout) וניתוב מעגלים (Routing)

**מעברי סינתזת מעגלים**
 - `AICliffordSynthesis`: סינתזה של מעגלי Clifford
 - `AILinearFunctionSynthesis`: סינתזה של מעגלי Linear Function
 - `AIPermutationSynthesis`: סינתזה של מעגלי Permutation
 - `AIPauliNetworkSynthesis`: סינתזה של מעגלי Pauli Network

כדי להשתמש במעברי ה-AI Transpiler, קודם [התקן את חבילת `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). בקר ב[תיעוד ה-API של qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) לקבלת מידע נוסף על האפשרויות השונות הזמינות.

## הרץ את מעברי ה-AI Transpiler מקומית או בענן
אם אתה רוצה להשתמש במעברי ה-Transpiler המבוססים על בינה מלאכותית בסביבה המקומית שלך בחינם, התקן את `qiskit-ibm-transpiler` עם כמה תלויות נוספות כך:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

ללא תלויות נוספות אלה, מעברי ה-Transpiler המבוססים על בינה מלאכותית רצים בענן דרך שירות Qiskit Transpiler Service (זמין רק למשתמשי IBM Quantum Premium Plan, Flex Plan, או On-Prem (דרך IBM Quantum Platform API) Plan). לאחר התקנת התלויות הנוספות, מצב ברירת המחדל להרצת מעברי ה-Transpiler המבוססים על בינה מלאכותית הוא שימוש במחשב המקומי שלך.

## מעבר AI Routing
המעבר `AIRouting` משמש גם כשלב פריסה (layout) וגם כשלב ניתוב (routing). ניתן להשתמש בו בתוך `PassManager` כך:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

כאן, ה-`backend` קובע עבור איזו מפת צימוד (coupling map) לנתב, ה-`optimization_level` (1, 2, או 3) קובע את מאמץ החישוב שיושקע בתהליך (ערך גבוה יותר בדרך כלל נותן תוצאות טובות יותר אך לוקח יותר זמן), וה-`layout_mode` מציין כיצד לטפל בבחירת הפריסה.
ל-`layout_mode` יש את האפשרויות הבאות:

- `keep`: זה מכבד את הפריסה שנקבעה על ידי מעברי ה-Transpiler הקודמים (או משתמש בפריסה הטריוויאלית אם לא נקבעה). בדרך כלל משתמשים בו רק כאשר המעגל חייב לרוץ על Qubits ספציפיים של המכשיר. לרוב מניב תוצאות גרועות יותר מכיוון שיש לו פחות מרחב לאופטימיזציה.
- `improve`: זה משתמש בפריסה שנקבעה על ידי מעברי ה-Transpiler הקודמים כנקודת התחלה. שימושי כשיש לך ניחוש ראשוני טוב לפריסה; לדוגמה, עבור מעגלים שנבנו בצורה שמשקפת בערך את מפת הצימוד של המכשיר. שימושי גם אם אתה רוצה לנסות מעברי פריסה ספציפיים אחרים בשילוב עם מעבר `AIRouting`.
- `optimize`: זהו המצב הברירת מחדל. עובד הכי טוב עבור מעגלים כלליים שאולי אין לך ניחושי פריסה טובים עבורם. מצב זה מתעלם מבחירות פריסה קודמות.
- `local_mode`: דגל זה קובע היכן רץ מעבר ה-`AIRouting`. אם `False`, ה-`AIRouting` רץ מרחוק דרך שירות Qiskit Transpiler Service. אם `True`, החבילה מנסה להריץ את המעבר בסביבה המקומית שלך עם נפילה חזרה למצב ענן אם התלויות הנדרשות לא נמצאות.

## מעברי סינתזת מעגלים AI
מעברי סינתזת המעגלים של ה-AI מאפשרים לך לאטב חתיכות של סוגי מעגלים שונים ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) על ידי סינתוז מחדש שלהם. דרך טיפוסית להשתמש במעבר הסינתזה היא כך:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

הסינתזה מכבדת את מפת הצימוד של המכשיר: ניתן להריץ אותה בבטחה לאחר מעברי ניתוב אחרים מבלי לשבש את המעגל, כך שהמעגל הכולל עדיין יציית למגבלות המכשיר. כברירת מחדל, הסינתזה תחליף את תת-המעגל המקורי רק אם תת-המעגל המסונתז משפר את המקור (כרגע בודק רק ספירת CNOT), אך ניתן לאלץ החלפה תמיד על ידי הגדרת `replace_only_if_better=False`.

מעברי הסינתזה הבאים זמינים מ-`qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: סינתזה עבור מעגלי [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (בלוקים של שערי `H`, `S`, ו-`CX`). כרגע עד תשעה בלוקי Qubit.
- *AILinearFunctionSynthesis*: סינתזה עבור מעגלי [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (בלוקים של שערי `CX` ו-`SWAP`). כרגע עד תשעה בלוקי Qubit.
- *AIPermutationSynthesis*: סינתזה עבור מעגלי [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (בלוקים של שערי `SWAP`). כרגע זמין עבור בלוקים של 65, 33, ו-27 Qubits.
- *AIPauliNetworkSynthesis*: סינתזה עבור מעגלי Pauli Network (בלוקים של שערי `H`, `S`, `SX`, `CX`, `RX`, `RY` ו-`RZ`). כרגע עד שישה בלוקי Qubit.

אנו צופים להגדיל בהדרגה את גודל הבלוקים הנתמכים.

כל המעברים משתמשים ב-thread pool כדי לשלוח מספר בקשות במקביל. כברירת מחדל, מספר ה-threads המקסימלי הוא מספר הליבות פלוס ארבע (ערכי ברירת מחדל של אובייקט ה-`ThreadPoolExecutor` בפייתון). עם זאת, ניתן להגדיר ערך משלך עם הארגומנט `max_threads` בזמן יצירת המעבר. לדוגמה, השורה הבאה יוצרת את מעבר `AILinearFunctionSynthesis`, המאפשרת לו להשתמש בלא יותר מ-20 threads.